# Projet de groupe Web scraping : véhicules d'occasion (Ottawa-Gatineau)

**Cours :** Collecte et nettoyage des données (projet UA2)
**Groupe :** Yacine Lezoul (2748526) et Mouncif Bouchachi (2743259)
**Activité :** concessionnaires de véhicules d'occasion de la région Ottawa-Gatineau

Ce notebook suit la démarche du document « Projet de groupe Web scraping » :

| Partie | Contenu | Où dans ce notebook |
|---|---|---|
| 0 | Préparation (environnement, User-Agent, délais) | Section 0 |
| 1 | Cadre légal et éthique (5 questions, robots.txt, conditions, API) | Section 1 |
| 2 | Diagnostic de chaque page (cas A à I) | Section 2 |
| 3 et 4 | Collecte et pipeline | À venir |
| 5 | Livrables : CSV, journal de tests, paragraphe par URL | À venir |

**Organisation des dossiers**

```
Projet_UA2_ETL/
├── notebook/
│   └── projet_vehicules.ipynb   # ce notebook
├── src/
│   ├── projet_scraping.py       # boîte à outils fournie par le cours
│   ├── scraper.py               # pipeline fourni par le cours
│   └── spider_livres.py         # exemple Scrapy fourni par le cours
├── pages_brutes/                # HTML téléchargé (non publié sur GitHub)
├── resultats/
│   └── journal_tests.csv        # journal de tests (annexe C)
├── requirements.txt             # versions des bibliothèques
└── .gitignore
```

## 0. Préparation (partie 0 du cours)

### 0.1 Chemins et importation de la boîte à outils

Le notebook est dans `notebook/` et la boîte à outils du cours dans `src/`. La cellule suivante
retrouve la racine du projet (le dossier qui contient `src/`), quel que soit le dossier
depuis lequel le notebook est lancé, puis importe `projet_scraping`.

In [ ]:
import csv
import json
import re
import sys
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd


def trouver_racine(depart=None):
    """Remonte les dossiers jusqu'à trouver celui qui contient src/projet_scraping.py."""
    dossier = Path(depart or Path.cwd()).resolve()
    for candidat in [dossier, *dossier.parents]:
        if (candidat / "src" / "projet_scraping.py").exists():
            return candidat
    raise FileNotFoundError("Dossier src/ introuvable : lancez le notebook depuis le projet.")


RACINE = trouver_racine()
DOSSIER_SRC = RACINE / "src"
DOSSIER_HTML = RACINE / "pages_brutes"
DOSSIER_RESULTATS = RACINE / "resultats"
FICHIER_JOURNAL = DOSSIER_RESULTATS / "journal_tests.csv"

if str(DOSSIER_SRC) not in sys.path:
    sys.path.insert(0, str(DOSSIER_SRC))

import projet_scraping as ws

DOSSIER_HTML.mkdir(exist_ok=True)
DOSSIER_RESULTATS.mkdir(exist_ok=True)
print("Racine du projet :", RACINE)

### 0.2 Version de Python

Le document « Avant de scraper » (section 3.1) précise que `urllib.robotparser` applique
correctement robots.txt (règle la plus longue, caractères `*` et `$`) seulement à partir de
**Python 3.13.14** ou **3.14.5**. Avec une version plus ancienne, une page interdite peut être
déclarée permise. On vérifie donc la version avant toute collecte.

In [ ]:
v = sys.version_info
version_ok = (v[:2] == (3, 13) and v >= (3, 13, 14)) or (v[:2] == (3, 14) and v >= (3, 14, 5)) or v[:2] > (3, 14)
print(f"Python {v.major}.{v.minor}.{v.micro} :", "conforme" if version_ok else "TROP ANCIENNE, mettre Python à jour")
assert version_ok, "Mettre Python à jour avant de continuer (voir « Avant de scraper », section 3.1)."

### 0.3 Réglages utilisés partout

| Réglage | Valeur | Pourquoi |
|---|---|---|
| `USER_AGENT` | nom du projet + nos deux courriels | Nous identifier honnêtement : un administrateur peut nous contacter au lieu de nous bloquer |
| `DELAI_MINIMUM` | 2 s | Le cours exige au moins 1 s entre deux requêtes au même site |
| Crawl-delay | lu dans robots.txt (5 s pour les sites D2C Media) | On applique toujours le plus grand des deux délais |
| `TIMEOUT` | 15 s (valeur de `projet_scraping.py`) | Abandonner une requête qui ne répond pas |

In [ ]:
COURRIEL_YACINE = "2748526@collegelacite.ca"
COURRIEL_MOUNCIF = "2743259@collegelacite.ca"

USER_AGENT = (
    "CoursCollecteDonnees-LaCite/1.0 (projet etudiant; "
    f"contact: {COURRIEL_YACINE}, {COURRIEL_MOUNCIF})"
)
DELAI_MINIMUM = 2.0

session = ws.creer_session(USER_AGENT)
print("User-Agent :", session.headers["User-Agent"])
print("Délai minimum :", DELAI_MINIMUM, "s ; timeout :", ws.TIMEOUT, "s")

## 1. Cadre légal et éthique (partie 1 du cours)

### 1.1 Les cinq questions avant toute collecte

| Question du cours | Notre réponse |
|---|---|
| 1. Existe-t-il une API officielle ou un jeu de données ouvert ? | Non. Aucun lien « API » ou « données » sur les sites, aucun jeu de données d'inventaire de concessionnaires sur ouvert.canada.ca ni sur donneesquebec.ca (Statistique Canada publie seulement des ventes agrégées). Il existe des fournisseurs commerciaux payants (MarketCheck, VinAudit), qui ne sont ni officiels ni ouverts. Vérification automatique complémentaire à la section 2. |
| 2. Que dit robots.txt ? | Lu pour chaque site (tableau 1.2), puis revérifié automatiquement par `robots_autorise()` avant chaque requête. |
| 3. Que disent les conditions d'utilisation ? | Lues pour chaque site. Les sites qui interdisent les robots ou le moissonnage sont exclus (tableau 1.3). |
| 4. Y a-t-il des renseignements personnels ? | Non : un inventaire décrit des produits (prix, année, kilométrage). On ne collecte ni noms ni téléphones de vendeurs. |
| 5. Mon script est-il poli ? | User-Agent identifiable, délai d'au moins 2 s (5 s si robots.txt le demande), une seule requête par page, arrêt immédiat en cas de blocage. |

**Contexte canadien.** Les sites de Gatineau relèvent de la Loi 25 (Québec) et ceux d'Ottawa de la
LPRPDE : aucune difficulté ici puisqu'aucun renseignement personnel n'est collecté. Pour le droit
d'auteur, le cours rappelle que les faits bruts (un prix, une date) ne sont pas protégés : nous
collectons des faits (prix, année, kilométrage), ni photos ni textes rédigés.

> **À refaire avant la remise :** revérifier la question 1 et mettre à jour la date ci-dessous.

In [ ]:
VERIFICATION_API_MANUELLE = (
    "Aucune API officielle ni donnée ouverte trouvée (pied de page du site, ouvert.canada.ca, "
    "donneesquebec.ca ; vérifié le 2026-10-09). Seules des sources commerciales payantes existent."
)

### 1.2 Les 18 sites validés

Pour chaque site : sa plateforme (les sites d'une même plateforme ont le même HTML), son
Crawl-delay relevé dans robots.txt, et les règles relevées dans robots.txt et dans les
conditions d'utilisation (vérifiées le 9 octobre 2026).

In [ ]:
SITES = [
    # --- Gatineau : plateforme Magnetis
    {"id": "hyundai_gatineau", "nom": "Hyundai Gatineau", "ville": "Gatineau", "province": "QC",
     "plateforme": "Magnetis", "delai_robots": None,
     "url": "https://hyundaigatineau.ca/en/pre-owned",
     "regles": "robots.txt : rien n'est bloqué. Conditions : usage personnel et non commercial."},
    {"id": "gatineau_acura", "nom": "Gatineau Acura", "ville": "Gatineau", "province": "QC",
     "plateforme": "Magnetis", "delai_robots": None,
     "url": "https://gatineauacura.com/en/pre-owned",
     "regles": "robots.txt : rien n'est bloqué. Conditions : usage personnel et non commercial."},
    {"id": "gatineau_kia", "nom": "Gatineau Kia", "ville": "Gatineau", "province": "QC",
     "plateforme": "Magnetis", "delai_robots": None,
     "url": "https://www.gatineaukia.ca/en/pre-owned",
     "regles": "robots.txt : rien n'est bloqué. Conditions : usage personnel et non commercial."},
    # --- Gatineau : plateforme D2C Media
    {"id": "dupont_honda", "nom": "Dupont & Dupont Honda", "ville": "Gatineau", "province": "QC",
     "plateforme": "D2C Media", "delai_robots": 5,
     "url": "https://www.dupontduponthonda.com/occasion/recherche.html",
     "regles": "robots.txt : Crawl-delay 5 s ; /used/detail.html et /json/ interdits (liste seulement). "
               "Conditions : usage personnel et non commercial."},
    {"id": "hull_hyundai", "nom": "Hull Hyundai", "ville": "Gatineau (Hull)", "province": "QC",
     "plateforme": "D2C Media", "delai_robots": 5,
     "url": "https://www.hullhyundai.com/occasion/recherche.html",
     "regles": "robots.txt : Crawl-delay 5 s ; /used/detail.html et /json/ interdits (liste seulement). "
               "Conditions : usage personnel et non commercial."},
    {"id": "hull_nissan", "nom": "Hull Nissan", "ville": "Gatineau (Hull)", "province": "QC",
     "plateforme": "D2C Media", "delai_robots": 5,
     "url": "https://www.hullnissangatineau.ca/occasion/recherche.html",
     "regles": "robots.txt : Crawl-delay 5 s ; /used/detail.html et /json/ interdits (liste seulement). "
               "Conditions : usage personnel et non commercial."},
    {"id": "buckingham_gm", "nom": "Buckingham Chevrolet Buick GMC", "ville": "Gatineau (Buckingham)",
     "province": "QC", "plateforme": "D2C Media", "delai_robots": 5,
     "url": "https://www.buckinghamgm.com/occasion/recherche.html",
     "regles": "robots.txt : Crawl-delay 5 s ; /used/detail.html et /json/* interdits (liste seulement). "
               "Conditions : usage personnel et non commercial."},
    # --- Ottawa : plateforme EDealer
    {"id": "ottawa_honda", "nom": "Ottawa Honda", "ville": "Ottawa", "province": "ON",
     "plateforme": "EDealer", "delai_robots": None,
     "url": "https://www.ottawahonda.com/inventory/used/",
     "regles": "robots.txt : /inventory/ permis ; URL avec filtres interdites sauf ?condition=USED. "
               "Aucune condition d'utilisation trouvée."},
    # --- Ottawa : groupe Dilawri
    {"id": "bank_street_mazda", "nom": "Bank Street Mazda", "ville": "Ottawa", "province": "ON",
     "plateforme": "Dilawri", "delai_robots": None,
     "url": "https://www.bankstreetmazda.com/ottawa-Mazda-dealer/used-cars/used-vehicle-inventory?type=U",
     "regles": "robots.txt : seuls /oops et /signalr/ bloqués. Aucune condition d'utilisation trouvée."},
    {"id": "bank_street_hyundai", "nom": "Bank Street Hyundai", "ville": "Ottawa", "province": "ON",
     "plateforme": "Dilawri", "delai_robots": None,
     "url": "https://www.bankstreethyundai.com/ottawa-gloucester-hyundai-dealer/used-cars/used-vehicle-inventory?type=U",
     "regles": "robots.txt : seuls /oops et /signalr/ bloqués. Aucune condition d'utilisation trouvée."},
    {"id": "bank_street_kia", "nom": "Bank Street Kia", "ville": "Ottawa", "province": "ON",
     "plateforme": "Dilawri", "delai_robots": None,
     "url": "https://www.bankstreetkia.com/ottawa-Kia-dealer/used-cars/used-vehicle-inventory?type=U",
     "regles": "robots.txt : seuls /oops et /signalr/ bloqués. Aucune condition d'utilisation trouvée."},
    {"id": "bank_street_mitsubishi", "nom": "Bank Street Mitsubishi", "ville": "Ottawa", "province": "ON",
     "plateforme": "Dilawri", "delai_robots": None,
     "url": "https://www.bankstreetmitsubishi.com/ottawa-Mitsubishi-dealer/used-cars/used-vehicle-inventory?type=U",
     "regles": "robots.txt : seuls /oops et /signalr/ bloqués. Aucune condition d'utilisation trouvée."},
    {"id": "barrhaven_mazda", "nom": "Barrhaven Mazda", "ville": "Ottawa (Barrhaven)", "province": "ON",
     "plateforme": "Dilawri", "delai_robots": None,
     "url": "https://www.barrhavenmazda.com/ottawa/used/inventory?type=U",
     "regles": "robots.txt : seuls /oops et /signalr/ bloqués. Aucune condition d'utilisation trouvée."},
    {"id": "barrhaven_cjdr", "nom": "Barrhaven Chrysler Jeep Dodge Ram", "ville": "Ottawa (Barrhaven)",
     "province": "ON", "plateforme": "Dilawri", "delai_robots": None,
     "url": "https://www.barrhavenjeepdodgeram.com/used-vehicles/used-vehicle-inventory?type=U",
     "regles": "robots.txt : seuls /oops et /signalr/ bloqués. Aucune condition d'utilisation trouvée."},
    {"id": "dilawri_cjdr", "nom": "Dilawri Chrysler Jeep Dodge Ram", "ville": "Ottawa", "province": "ON",
     "plateforme": "Dilawri", "delai_robots": None,
     "url": "https://www.dilawrichrysler.com/chrysler-jeep-dodge-ram-dealer-ottawa/used-cars/used-vehicle-inventory?type=U",
     "regles": "robots.txt : seuls /oops et /signalr/ bloqués. Aucune condition d'utilisation trouvée."},
    {"id": "orleans_mitsubishi", "nom": "Orleans Mitsubishi", "ville": "Ottawa (Orléans)", "province": "ON",
     "plateforme": "Dilawri", "delai_robots": None,
     "url": "https://www.orleansmitsubishi.com/mitsubishi/used-inventory.html?type=U",
     "regles": "robots.txt : /oops, /signalr/ et le blogue bloqués. Aucune condition d'utilisation trouvée."},
    {"id": "hyundai_hunt_club", "nom": "Hyundai On Hunt Club", "ville": "Ottawa (Nepean)", "province": "ON",
     "plateforme": "Dilawri", "delai_robots": None,
     "url": "https://www.hyundaionhuntclub.com/hyundai/used~vehicles/inventory.html?type=U",
     "regles": "robots.txt : seuls /oops et /signalr/ bloqués. Aucune condition d'utilisation trouvée."},
    {"id": "kia_hunt_club", "nom": "Kia On Hunt Club", "ville": "Ottawa (Nepean)", "province": "ON",
     "plateforme": "Dilawri", "delai_robots": None,
     "url": "https://www.kiaonhuntclub.com/kia/used-inventory.html?type=U",
     "regles": "robots.txt : /oops, /signalr/ et le blogue bloqués. Aucune condition d'utilisation trouvée."},
]

pd.DataFrame(SITES)[["nom", "ville", "plateforme", "delai_robots", "regles"]]

### 1.3 Règle du groupe : zéro risque éthique

**En cas de doute sur la légalité ou l'éthique d'une collecte, on ne collecte pas.** Tout ce qui
est interdit, ou dont on ne peut pas être sûrs que c'est permis, est évité et noté dans le journal.

| Niveau | Sites ou pratiques | Raison |
|---|---|---|
| 🔴 **Interdit, on n'y touche pas** | Toyota Gatineau, Villa Toyota | Conditions SM360 : robots interdits |
| | Bel-Air Toyota, Car Canada, groupe Myers (16 concessionnaires) | Conditions : « spider, crawl, or scrape » interdits |
| | Ottawa Honda | Protection Cloudflare (403) : la contourner est interdit |
| | Pages détail et chemin `/json/` des sites D2C Media | Interdits par robots.txt |
| 🟠 **Évité par prudence** | Fichier `inventory-index.json` de Buckingham GM (cas E1b) | Fichier interne du chatbot : pas interdit mot à mot par robots.txt, mais on ne peut pas être sûrs que son usage par un robot est voulu |
| | Playwright pour charger le reste de l'inventaire D2C (cas E2) | Reviendrait à imiter un navigateur pour obtenir ce que le site ne sert pas aux robots |
| | Véhicules D2C absents du HTML (Buckingham 25, Hull Hyundai environ 9, Dupont 2) | Seule voie d'accès : JavaScript ou chemins interdits |
| | Téléphones, noms de vendeurs, descriptions rédigées, photos | Renseignements de contact et contenu protégé : on ne garde que des faits bruts |
| 🟢 **Permis** | HTML des listes d'inventaire des 17 sites accessibles | robots.txt le permet, aucune condition n'interdit les robots |
| | JSON-LD intégré dans ces pages (cas E1a) | Fait partie de la page déjà téléchargée, aucune requête de plus |
| | Pages suivantes par `?vip=` et `?pindex=` | Liens présents dans le HTML, non bloqués par robots.txt |

**Réserve.** Les sites Magnetis (3) et D2C Media (4) limitent l'usage du contenu à un « usage
personnel et non commercial ». Notre usage est scolaire, non commercial et non publié, et ces sites
ont été validés par le professeur. Une confirmation écrite du professeur est demandée pour lever
tout doute ; sans elle, seuls les 10 sites Dilawri (aucune condition d'utilisation) seraient conservés.

### 1.4 Sites écartés avant toute collecte

Ces sites ont été examinés puis écartés parce que leurs conditions d'utilisation interdisent la
collecte automatisée. Conformément aux consignes, ils sont notés dans le journal mais ne comptent
pas parmi nos sites. Aucune requête automatisée ne leur est envoyée.

In [ ]:
SITES_EXCLUS = pd.DataFrame([
    {"site": "Toyota Gatineau", "url": "https://www.toyotagatineau.ca/fr/conditions-utilisation",
     "motif": "Conditions (plateforme SM360) : interdit « d'utiliser un robot, ou tout autre appareil, "
              "processus ou moyen automatique pour accéder au site Web »."},
    {"site": "Villa Toyota", "url": "https://www.villatoyota.com/fr/conditions-utilisation",
     "motif": "Mêmes conditions SM360 : robots interdits."},
    {"site": "Bel-Air Toyota", "url": "https://www.belairtoyota.ca/terms-conditions",
     "motif": "Conditions, section 12 : interdit de « spider, crawl, or scrape »."},
    {"site": "Car Canada", "url": "https://www.carcanada.com/terms-conditions",
     "motif": "Conditions, section 12 : interdit de « spider, crawl, or scrape »."},
    {"site": "Groupe Myers (16 concessionnaires)", "url": "https://www.myers.ca/terms-conditions",
     "motif": "Conditions, section 12 : interdit de « spider, crawl, or scrape »."},
])
SITES_EXCLUS

### 1.5 Liste des URL à traiter, sans doublons

Avant toute requête, on construit la liste des URL à traiter en retirant les doublons. Deux URL
sont considérées identiques si elles ne diffèrent que par la casse du domaine ou par une barre
oblique finale.

In [ ]:
def normaliser_url(url):
    """Clé de comparaison : domaine en minuscules, sans barre oblique finale."""
    p = urlparse(url.strip())
    chemin = p.path.rstrip("/") or "/"
    return f"{p.scheme.lower()}://{p.netloc.lower()}{chemin}" + (f"?{p.query}" if p.query else "")


def sites_sans_doublons(sites):
    """Garde la première occurrence de chaque URL et signale les doublons retirés."""
    vues, uniques = set(), []
    for site in sites:
        cle = normaliser_url(site["url"])
        if cle in vues:
            print(f"Doublon retiré : {site['id']} ({site['url']})")
            continue
        vues.add(cle)
        uniques.append(site)
    return uniques


SITES_A_TRAITER = sites_sans_doublons(SITES)
print(f"{len(SITES)} sites déclarés, {len(SITES_A_TRAITER)} URL uniques à traiter.")

## 2. Diagnostic (partie 2 du cours)

Pour chaque URL, on applique l'arbre de décision du cours :

1. `ws.diagnostiquer(url)` lit robots.txt, envoie **une** requête et observe la page :
   statut HTTP, protection anti-robot, encodage, JavaScript, JSON intégré, tableaux,
   pagination, blocs répétés. Il en déduit le ou les cas (A à I).
2. Si le cas est **STOP** (robots.txt) ou **G** (blocage), on s'arrête : on ne contourne jamais.
3. Sinon, on télécharge la page une fois avec `ws.obtenir()` (qui revérifie robots.txt, gère
   les erreurs 429 et 5xx, et s'arrête sur un 403) et on l'enregistre dans `pages_brutes/`.
4. On cherche les signes d'**API** ou de données structurées (question 1 de la partie 1, cas E) :
   JSON-LD, scripts `application/json`, adresses d'API dans le code, et permission robots.txt
   pour les chemins `/api/` et `/json/`.

**Rappel des cas du cours**

| Cas | Situation | Technique |
|---|---|---|
| A | HTML statique | requests + BeautifulSoup |
| B | Tableau HTML | pandas.read_html |
| C | Pagination | Boucle + délai + arrêt |
| D | Liste vers pages détail | Collecte en deux niveaux |
| E | Contenu JavaScript | E1a JSON intégré, E1b API cachée (F12 > Réseau), E2 Playwright |
| F | Connexion | Session + jeton CSRF |
| G | Erreurs et blocages | Réessayer (429, 5xx) ou s'arrêter (403, 402) |
| H | Encodage, texte sale | Corriger, normaliser |
| I | Grand volume | Scrapy |

### 2.1 Recherche d'API et de données structurées

In [ ]:
CHEMINS_API = ["/api/", "/json/"]
_MOTIF_API = re.compile(r"""["'](/[\w\-/~.]*?(?:api|json|graphql)[\w\-/~.]*)["']""", re.I)
_FICHIERS_STATIQUES = re.compile(r"\.(js|css|png|jpe?g|gif|svg|webp|woff2?)$", re.I)


def chercher_api(html, url, session):
    """
    Observe la page déjà téléchargée (aucune requête de plus) :
      - types JSON-LD et nombre de véhicules décrits (schema.org Vehicle, Car, ItemList) ;
      - scripts <script type="application/json"> ;
      - adresses d'API ou de JSON citées dans le code ;
      - permission robots.txt pour /api/ et /json/ (robots.txt est déjà en cache).
    """
    sp = ws.soupe(html)

    types, nb_vehicules = [], 0
    for script in sp.select("script[type='application/ld+json']"):
        try:
            donnees = json.loads(script.string or "")
        except (json.JSONDecodeError, TypeError):
            types.append("JSON-LD illisible")
            continue
        elements = donnees if isinstance(donnees, list) else [donnees]
        # Certains sites regroupent leurs objets dans un « @graph » : on les déplie
        elements = [x for e in elements for x in (e.get("@graph", [e]) if isinstance(e, dict) else [e])]
        for element in elements:
            if not isinstance(element, dict):
                continue
            t = element.get("@type")
            t = "/".join(t) if isinstance(t, list) else str(t)
            if t == "ItemList":
                n = len(element.get("itemListElement", []))
                nb_vehicules += n
                types.append(f"ItemList({n})")
            else:
                if t in ("Vehicle", "Car"):
                    nb_vehicules += 1
                types.append(t)

    resume_types = ", ".join(f"{t} x{types.count(t)}" for t in dict.fromkeys(types))
    adresses = sorted({a for a in _MOTIF_API.findall(html) if not _FICHIERS_STATIQUES.search(a)})

    p = urlparse(url)
    base = f"{p.scheme}://{p.netloc}"
    permissions = []
    for chemin in CHEMINS_API:
        ok, _, _ = ws.robots_autorise(base + chemin, session.headers["User-Agent"], session)
        permissions.append(f"{chemin} {'permis' if ok else 'INTERDIT'}")

    return {
        "json_ld_types": resume_types,
        "json_ld_vehicules": nb_vehicules,
        "scripts_application_json": len(sp.select("script[type='application/json']")),
        "adresses_api_dans_page": " | ".join(adresses[:10]),
        "robots_chemins_api": " ; ".join(permissions),
    }

### 2.2 Diagnostic d'un site

In [ ]:
def diagnostiquer_site(site, session):
    """Diagnostic + arrêt si nécessaire + sauvegarde du HTML + recherche d'API. Retourne une ligne du journal."""
    print(f"\n{'=' * 70}\n{site['nom']} ({site['ville']}, {site['plateforme']})\n{'=' * 70}")

    ligne = {
        "url": site["url"], "cas_confirme": "", "nb_lignes": "",
        "date": datetime.now().isoformat(timespec="seconds"),
        "id": site["id"], "nom": site["nom"], "ville": site["ville"],
        "province": site["province"], "plateforme": site["plateforme"],
        "regles_site": site["regles"],
        "api_officielle_ou_donnees_ouvertes": VERIFICATION_API_MANUELLE,
        "fichier_html": "", "problemes_remarques": "",
    }
    remarques = []

    # --- Diagnostic (robots.txt + 1 requête)
    d = ws.diagnostiquer(site["url"], session=session, afficher=True)
    crawl_delay = d.get("crawl_delay")
    delai = max(DELAI_MINIMUM, crawl_delay or 0, site["delai_robots"] or 0)
    ligne.update(
        statut=d.get("statut"), robots=d.get("robots_raison"), crawl_delay=crawl_delay,
        delai_utilise_s=delai, cas_diagnostic=" | ".join(d.get("cas", [])),
        recommandation=d.get("recommandation", ""), technique=d.get("recommandation", ""),
        protection=" | ".join(d.get("protection", [])), signes_js=" | ".join(d.get("signes_js", [])),
        json_integre=" | ".join(d.get("json_integre", [])), blocs_repetes=d.get("blocs_repetes", ""),
        pagination=d.get("pagination", ""), page_suivante=d.get("page_suivante") or "",
        nb_tableaux=d.get("nb_tableaux", ""), taille_ko=d.get("taille_ko", ""),
        encodage=f"déclaré={d.get('charset_declare')} ; détecté={d.get('encodage_detecte')}",
        alertes=" | ".join(d.get("alertes", [])),
    )
    time.sleep(delai)    # diagnostiquer() n'attend pas après sa requête : on respecte le délai ici

    # --- Arrêts (arbre de décision)
    if d.get("statut") is None:
        ligne.update(technique="Aucune (site injoignable)",
                     problemes_remarques=f"Site injoignable ({ligne['recommandation']}). Relancer plus tard.")
        return ligne
    cas = ligne["cas_diagnostic"]
    if cas.startswith("STOP") or cas.startswith("G"):
        print(f"→ Arrêt pour {site['nom']} : {cas}")
        ligne.update(technique="Aucune (arrêt volontaire)",
                     problemes_remarques=f"ARRÊT : {cas} (statut {d.get('statut')}). Aucune tentative de contournement.")
        return ligne

    # --- Sauvegarde du HTML
    try:
        r = ws.obtenir(session, site["url"], delai=delai)
    except ws.AccesBloque as e:
        ligne["problemes_remarques"] = f"ARRÊT (accès bloqué) : {e}"
        return ligne
    except Exception as e:
        ligne["problemes_remarques"] = f"ÉCHEC : {type(e).__name__} : {e}"
        return ligne
    if r is None:
        ligne["problemes_remarques"] = "Page introuvable (404)."
        return ligne

    chemin = DOSSIER_HTML / f"{site['id']}.html"
    chemin.write_text(r.text, encoding="utf-8")
    ligne["fichier_html"] = str(chemin.relative_to(RACINE))
    print(f"→ HTML enregistré : {ligne['fichier_html']} ({len(r.content) / 1024:.0f} Ko)")

    # --- API et données structurées
    api = chercher_api(r.text, site["url"], session)
    ligne.update(api)
    print(f"→ JSON-LD : {api['json_ld_types'] or 'aucun'} ; véhicules décrits : {api['json_ld_vehicules']}")
    print(f"→ robots.txt pour les API : {api['robots_chemins_api']}")
    if "INTERDIT" in api["robots_chemins_api"]:
        remarques.append("Une API interne existe peut-être, mais robots.txt interdit son chemin : ne pas l'utiliser.")

    ligne["problemes_remarques"] = " ".join(remarques)
    return ligne

### 2.3 Lancement sur toutes les URL

Environ 3 minutes : 2 requêtes par site, avec le délai entre chaque.

In [ ]:
lignes_journal = []
for site in SITES_A_TRAITER:
    try:
        lignes_journal.append(diagnostiquer_site(site, session))
    except Exception as e:     # un site en erreur ne doit pas arrêter les autres
        lignes_journal.append({
            "url": site["url"], "date": datetime.now().isoformat(timespec="seconds"),
            "id": site["id"], "nom": site["nom"],
            "problemes_remarques": f"ERREUR INATTENDUE : {type(e).__name__} : {e}",
        })
print(f"\n{len(lignes_journal)} sites traités.")

### 2.4 Résumé du diagnostic

In [ ]:
journal = pd.DataFrame(lignes_journal)
# reindex : affiche les colonnes même si aucun site n'a pu être téléchargé
journal.reindex(columns=["nom", "plateforme", "statut", "cas_diagnostic", "blocs_repetes",
                         "json_ld_vehicules", "robots_chemins_api"])

### 2.5 Du diagnostic automatique au cas confirmé

Le cours demande de confirmer chaque diagnostic en ouvrant la page dans le navigateur (F12).
Vérifications faites le 9 octobre 2026 dans Microsoft Edge :

- **Hyundai Gatineau** : 2 pages (`?vip=2`) ; Ctrl+U montre un JSON-LD `ItemList` de 20 véhicules.
- **Bank Street Mazda** : la liste se charge au défilement dans le navigateur, mais le HTML contient
  des liens `?pindex=` qui donnent les mêmes pages sans JavaScript.
- **Ottawa Honda** : la page s'ouvre lentement dans le navigateur (vérification Cloudflare
  silencieuse), alors que notre script reçoit un 403.
- **Buckingham GM et Hull Hyundai** : capture de l'onglet Réseau (fichier HAR). Aucune requête ne
  charge de véhicules au défilement. Chez Buckingham, la page charge le fichier interne du chatbot
  `inventory-index.json` (61 véhicules d'occasion et démos), **non utilisé par prudence** (section 1.3).

Les autres sites ont la même structure HTML que le site vérifié de leur plateforme ; leur cas
repose sur ce site de référence et sur le HTML enregistré.

**Les cas du cours appliqués à nos sites**

| Cas | Ce que ça veut dire | Sites concernés | Difficulté rencontrée | Ce qu'on fait |
|---|---|---|---|---|
| **A** HTML statique | Les véhicules sont dans le code HTML | Les 17 sites accessibles | Chaque plateforme a ses propres balises | Sélecteurs CSS adaptés à chaque plateforme |
| **B** Tableau HTML | Données dans un `<table>` | Aucun (fausse alerte sur 5 sites Dilawri et Hull Hyundai) | Le diagnostic a pris des tableaux de calcul de paiement pour des données | Cas écarté |
| **C** Pagination | Inventaire réparti sur plusieurs pages | Hyundai Gatineau, Gatineau Kia (`?vip=`), 10 sites Dilawri (`?pindex=`) | Défilement continu dans le navigateur ; liens de page non vus par le diagnostic | On suit les liens de page du HTML, avec délai et arrêt |
| **D** Liste vers pages détail | Une page par véhicule | Possible partout, interdit chez D2C Media | Une requête de plus par véhicule | Non utilisé : la liste suffit, et c'est plus poli |
| **E1a** JSON intégré | Données en JSON dans la page | Magnetis (`ItemList`), D2C Media (`Vehicle`) | Le JSON-LD ne contient pas le kilométrage | JSON-LD + HTML combinés, sans requête de plus |
| **E1b** API cachée | La page appelle un fichier JSON | Buckingham GM (`inventory-index.json`) | Fichier interne du chatbot, autorisation incertaine | Non utilisé par prudence |
| **E2** Navigateur automatisé | Playwright exécute le JavaScript | Aurait donné tout l'inventaire D2C | Revient à imiter un navigateur | Non utilisé |
| **F** Connexion | Compte requis | Aucun | | |
| **G** Blocage | Le site refuse l'accès | Ottawa Honda (403, Cloudflare) | Le site s'ouvre dans un vrai navigateur, pas pour notre script | Arrêt, aucun contournement |
| **H** Encodage | Accents mal décodés | 4 sites D2C Media (iso-8859-1) | Risque de « Ã© » au lieu de « é » | `corriger_encodage()` du cours |
| **I** Grand volume | Scrapy | Aucun | Environ 60 pages et 900 véhicules | Une boucle requests avec délai suffit |

In [ ]:
# Cas confirmés après vérification dans le navigateur (Microsoft Edge, F12), le 2026-10-09
# Règle du groupe : en cas de doute sur la légalité ou l'éthique, on ne collecte pas.
# Cas I (Scrapy) non retenu partout : volume faible (environ 60 pages), une boucle requests avec délai suffit.
CAS_CONFIRMES = {
    # --- Magnetis
    "hyundai_gatineau": "A + C + E1a (vérifié : 2 pages, lien ?vip=2 ; Ctrl+U : JSON-LD ItemList de "
                        "20 véhicules par page ; pages détail non utilisées, la liste suffit)",
    "gatineau_acura":   "A + E1a (même plateforme que Hyundai Gatineau ; une seule page, 29 fiches et "
                        "ItemList de 29 véhicules dans le HTML)",
    "gatineau_kia":     "A + C + E1a (même plateforme que Hyundai Gatineau ; lien ?vip=2, ItemList de "
                        "20 véhicules par page)",
    # --- D2C Media : seulement les fiches présentes dans le HTML
    "dupont_honda":     "A + H (36 fiches dans le HTML sur 38 annoncées ; les 2 autres ne sont accessibles "
                        "que par JavaScript ou par des chemins interdits : non collectées ; encodage iso-8859-1)",
    "hull_hyundai":     "A + H (HAR : aucune requête au défilement ; 39 fiches dans le HTML, seules collectées ; "
                        "pages détail et /json/ interdits par robots.txt ; encodage iso-8859-1)",
    "hull_nissan":      "A + H (12 fiches dans le HTML sur 12 annoncées : inventaire complet ; encodage iso-8859-1)",
    "buckingham_gm":    "A + H (HAR : 36 fiches dans le HTML, aucune requête au défilement ; la page charge "
                        "/js/json/chatboost/inventory/inventory-index.json (61 occasions et démos) : fichier "
                        "interne du chatbot, NON UTILISÉ PAR PRUDENCE car son usage par un robot n'est pas "
                        "clairement autorisé ; seules les 36 fiches du HTML sont collectées)",
    # --- EDealer
    "ottawa_honda":     "G (403 + défi Cloudflare « Just a moment » ; dans le navigateur, chargement lent = "
                        "vérification silencieuse ; ARRÊT, aucune tentative de contournement, site non collecté)",
    # --- Dilawri : pagination par les liens du HTML, sans JavaScript
    "bank_street_mazda":      "A + C (vérifié : défilement continu dans le navigateur, mais liens ?pindex= "
                              "présents dans le HTML ; on suit ces liens ; pages détail non utilisées)",
    "bank_street_hyundai":    "A + C (même plateforme que Bank Street Mazda ; liens ?pindex= dans le HTML)",
    "bank_street_kia":        "A + C (même plateforme ; liens ?pindex= ; tableaux détectés = calcul de "
                              "paiement : cas B écarté)",
    "bank_street_mitsubishi": "A + C (même plateforme ; liens ?pindex= ; cas B écarté, même raison)",
    "barrhaven_mazda":        "A + C (même plateforme ; liens ?pindex= dans le HTML)",
    "barrhaven_cjdr":         "A + C (même plateforme ; liens ?pindex= dans le HTML)",
    "dilawri_cjdr":           "A + C (même plateforme ; liens ?pindex= dans le HTML)",
    "orleans_mitsubishi":     "A + C (même plateforme ; liens ?pindex= ; cas B écarté, même raison)",
    "hyundai_hunt_club":      "A + C (même plateforme ; liens ?pindex= dans le HTML)",
    "kia_hunt_club":          "A + C (même plateforme ; liens ?pindex= ; cas B écarté ; JSON-LD illisible, "
                              "sans véhicules)",
}

# Technique retenue pour la collecte (colonne « technique » de l'annexe C)
TECHNIQUES = {
    "Magnetis":  "requests + BeautifulSoup + JSON-LD intégré ; pages ?vip= avec délai",
    "D2C Media": "requests + BeautifulSoup + JSON-LD intégré ; correction de l'encodage ; liste seulement",
    "Dilawri":   "requests + BeautifulSoup ; pages ?pindex= avec délai",
    "EDealer":   "Aucune (arrêt volontaire)",
}

journal["cas_confirme"] = journal["id"].map(CAS_CONFIRMES).fillna("")
journal["technique"] = journal["plateforme"].map(TECHNIQUES).fillna(journal["technique"])
journal.reindex(columns=["nom", "cas_diagnostic", "cas_confirme", "technique"])

### 2.6 Enregistrement du journal de tests (annexe C)

Les six colonnes de l'annexe C viennent en premier, puis les détails du diagnostic.
Encodage `utf-8-sig`, comme `scraper.py`, pour une lecture correcte dans Excel.

In [ ]:
COLONNES = [
    # Annexe C
    "url", "cas_diagnostic", "cas_confirme", "technique", "nb_lignes", "problemes_remarques",
    # Identification
    "date", "id", "nom", "ville", "province", "plateforme",
    # Partie 1 : cadre légal et éthique
    "robots", "crawl_delay", "delai_utilise_s", "regles_site",
    "api_officielle_ou_donnees_ouvertes", "robots_chemins_api",
    # Partie 2 : diagnostic
    "statut", "recommandation", "protection", "signes_js",
    "json_integre", "json_ld_types", "json_ld_vehicules", "scripts_application_json",
    "adresses_api_dans_page", "blocs_repetes", "pagination", "page_suivante", "nb_tableaux",
    "taille_ko", "encodage", "alertes", "fichier_html",
]

journal.reindex(columns=COLONNES).to_csv(FICHIER_JOURNAL, index=False, encoding="utf-8-sig")
print("Journal enregistré :", FICHIER_JOURNAL.relative_to(RACINE))

## Prochaines étapes

- Obtenir la confirmation écrite du professeur pour les sites Magnetis et D2C Media (section 1.3).
- Section 3 : collecte avec une seule fonction d'extraction, appelée une fois par URL de la liste
  dédoublonnée, et dédoublonnage des véhicules par NIV.
- Section 4 : nettoyage (prix, kilométrage, encodage) et fusion en un seul CSV.
- Section 5 : un paragraphe par URL (cas détecté, technique choisie, difficultés rencontrées).